# Exploratory Data Analysis: Pittsburgh Cooling Centers

This notebook explores the three core datasets used in the cooling center optimization model:
1. **Block Groups** - 1,050 Census block groups with demographics and vulnerability scores
2. **Candidate Sites** - 87 public facilities eligible for cooling center placement
3. **Scenario Results** - Optimization outputs across 5 objectives and 11 budget levels

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

DATA = Path('../data')

## 1. Block Group Demographics

In [ ]:
bg = gpd.read_file(DATA / 'processed' / 'block_groups.geojson')
print(f'Block groups: {len(bg)}')
print(f'Total population: {bg["total_pop"].sum():,.0f}')
bg[['total_pop', 'pct_65_plus', 'poverty_rate', 'pct_no_vehicle', 'vulnerability_score']].describe().round(2)

### 1.1 Population Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(bg['total_pop'], bins=40, color='#2563eb', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Population')
axes[0].set_ylabel('Block Groups')
axes[0].set_title('Population per Block Group')
axes[0].axvline(bg['total_pop'].median(), color='#B33951', ls='--', label=f'Median: {bg["total_pop"].median():.0f}')
axes[0].legend()

axes[1].hist(bg['pct_65_plus'], bins=40, color='#d97706', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('% Population 65+')
axes[1].set_ylabel('Block Groups')
axes[1].set_title('Senior Population Share')
axes[1].axvline(bg['pct_65_plus'].median(), color='#B33951', ls='--', label=f'Median: {bg["pct_65_plus"].median():.1f}%')
axes[1].legend()

axes[2].hist(bg['poverty_rate'], bins=40, color='#059669', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Poverty Rate (%)')
axes[2].set_ylabel('Block Groups')
axes[2].set_title('Poverty Rate Distribution')
axes[2].axvline(bg['poverty_rate'].median(), color='#B33951', ls='--', label=f'Median: {bg["poverty_rate"].median():.1f}%')
axes[2].legend()

plt.tight_layout()
plt.show()

### 1.2 Vulnerability Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(bg['vulnerability_score'], bins=40, color='#B33951', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Vulnerability Score')
axes[0].set_ylabel('Block Groups')
axes[0].set_title('Vulnerability Score Distribution')
axes[0].axvline(bg['vulnerability_score'].median(), color='#171717', ls='--', label=f'Median: {bg["vulnerability_score"].median():.2f}')
axes[0].axvline(bg['vulnerability_score'].quantile(0.9), color='#d97706', ls='--', label=f'P90: {bg["vulnerability_score"].quantile(0.9):.2f}')
axes[0].legend()

axes[1].hist(bg['pct_no_vehicle'], bins=40, color='#7c3aed', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('% Households Without Vehicle')
axes[1].set_ylabel('Block Groups')
axes[1].set_title('Vehicle Access Distribution')
axes[1].axvline(bg['pct_no_vehicle'].median(), color='#B33951', ls='--', label=f'Median: {bg["pct_no_vehicle"].median():.1f}%')
axes[1].legend()

plt.tight_layout()
plt.show()

### 1.3 Correlation Between Vulnerability Indicators

In [ ]:
indicators = ['pct_65_plus', 'poverty_rate', 'pct_no_vehicle', 'vulnerability_score', 'total_pop']
corr = bg[indicators].corr().round(2)

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, cmap='RdYlGn_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=1, ax=ax,
            xticklabels=['Age 65+', 'Poverty', 'No Vehicle', 'Vulnerability', 'Population'],
            yticklabels=['Age 65+', 'Poverty', 'No Vehicle', 'Vulnerability', 'Population'])
ax.set_title('Correlation Between Vulnerability Indicators')
plt.tight_layout()
plt.show()

### 1.4 Vulnerability Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
bg.plot(column='vulnerability_score', cmap='YlOrRd', legend=True, ax=ax,
        legend_kwds={'label': 'Vulnerability Score', 'shrink': 0.6})
ax.set_title('Vulnerability Score by Block Group', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

### 1.5 Top 10 Most Vulnerable Block Groups

In [ ]:
top_vuln = bg.nlargest(10, 'vulnerability_score')[['GEOID', 'total_pop', 'pct_65_plus', 'poverty_rate', 'pct_no_vehicle', 'vulnerability_score']].copy()
top_vuln = top_vuln.round(2)
top_vuln

---
## 2. Candidate Sites

In [ ]:
sites = gpd.read_file(DATA / 'processed' / 'candidate_sites.geojson')
print(f'Total candidate sites: {len(sites)}')
print(f'Existing cooling centers: {sites["is_existing_center"].sum()}')
print(f'Potential new sites: {(~sites["is_existing_center"]).sum()}')
print()
sites['type'].value_counts()

### 2.1 Site Types

In [ ]:
type_counts = sites['type'].value_counts()
colors = ['#2563eb' if t == 'Senior Center (Existing)' else '#91C7B1' for t in type_counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
type_counts.plot.barh(ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Number of Sites')
ax.set_title('Candidate Sites by Facility Type')
ax.invert_yaxis()
for i, v in enumerate(type_counts):
    ax.text(v + 0.3, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Candidate Sites Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
bg.plot(color='#e8e4dc', edgecolor='#c8c4bc', linewidth=0.3, ax=ax)

new_sites = sites[~sites['is_existing_center']]
existing = sites[sites['is_existing_center']]

new_sites.plot(ax=ax, color='#91C7B1', markersize=30, edgecolor='white', linewidth=0.5, label='Candidate Sites', zorder=3)
existing.plot(ax=ax, color='#2563eb', markersize=80, edgecolor='white', linewidth=1, marker='*', label='Existing Centers', zorder=4)

ax.set_title('Candidate Sites and Existing Cooling Centers', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_axis_off()
plt.tight_layout()
plt.show()

---
## 3. Distance Matrix Analysis

In [ ]:
dist = pd.read_csv(DATA / 'processed' / 'distance_matrix.csv', index_col='GEOID')
dist.columns = dist.columns.astype(int)
print(f'Distance matrix shape: {dist.shape}')
print(f'Min distance: {dist.values.min():.0f} m')
print(f'Max distance: {dist.values.max():.0f} m')
print(f'Mean distance: {dist.values.mean():.0f} m')

### 3.1 Nearest Site Distance (Existing Centers Only)

In [ ]:
existing_ids = sites[sites['is_existing_center']]['site_id'].values.astype(int)
nearest_existing = dist[existing_ids].min(axis=1) / 1000  # to km

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(nearest_existing, bins=50, color='#B33951', edgecolor='white', alpha=0.8)
ax.axvline(1.25, color='#059669', ls='--', lw=2, label='15-min walk (1.25 km)')
ax.axvline(nearest_existing.quantile(0.9), color='#d97706', ls='--', lw=2,
           label=f'P90: {nearest_existing.quantile(0.9):.1f} km')
ax.set_xlabel('Distance to Nearest Existing Center (km)')
ax.set_ylabel('Block Groups')
ax.set_title('Current Access: Distance to Nearest Existing Cooling Center')
ax.legend()
plt.tight_layout()
plt.show()

pct_covered = (nearest_existing <= 1.25).mean() * 100
print(f'Block groups within 15-min walk of existing center: {pct_covered:.1f}%')
print(f'Mean distance to nearest existing center: {nearest_existing.mean():.1f} km')
print(f'Max distance to nearest existing center: {nearest_existing.max():.1f} km')

---
## 4. Optimization Results

In [ ]:
sc = pd.read_csv(DATA / 'output' / 'scenario_results.csv')
print(f'Scenarios: {len(sc)}')
sc.head()

### 4.1 Mean Distance by Objective and Budget

In [ ]:
obj_labels = {
    'min_total_distance': 'Total Dist.',
    'min_pop_weighted': 'Pop-Weighted',
    'min_vuln_weighted': 'Vuln-Weighted',
    'min_worst_case': 'Worst-Case',
    'max_coverage': 'Coverage'
}
obj_colors = {
    'min_total_distance': '#2563eb',
    'min_pop_weighted': '#059669',
    'min_vuln_weighted': '#B33951',
    'min_worst_case': '#7c3aed',
    'max_coverage': '#d97706'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for obj in sc['objective'].unique():
    sub = sc[sc['objective'] == obj].sort_values('n_new')
    axes[0].plot(sub['n_new'], sub['mean_distance'] / 1000, marker='o', markersize=5,
                 label=obj_labels[obj], color=obj_colors[obj], linewidth=2)
axes[0].set_xlabel('Number of New Centers')
axes[0].set_ylabel('Mean Distance (km)')
axes[0].set_title('Mean Walking Distance')
axes[0].legend(fontsize=8)
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(1))

for obj in sc['objective'].unique():
    sub = sc[sc['objective'] == obj].sort_values('n_new')
    axes[1].plot(sub['n_new'], sub['max_distance'] / 1000, marker='o', markersize=5,
                 label=obj_labels[obj], color=obj_colors[obj], linewidth=2)
axes[1].set_xlabel('Number of New Centers')
axes[1].set_ylabel('Max Distance (km)')
axes[1].set_title('Worst-Case Distance')
axes[1].legend(fontsize=8)
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(1))

for obj in sc['objective'].unique():
    sub = sc[sc['objective'] == obj].sort_values('n_new')
    axes[2].plot(sub['n_new'], sub['pct_covered_15min'], marker='o', markersize=5,
                 label=obj_labels[obj], color=obj_colors[obj], linewidth=2)
axes[2].set_xlabel('Number of New Centers')
axes[2].set_ylabel('Coverage (%)')
axes[2].set_title('15-Min Walk Coverage')
axes[2].legend(fontsize=8)
axes[2].xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.suptitle('Scenario Analysis: Impact of Adding New Cooling Centers', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.2 Diminishing Returns Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for obj in sc['objective'].unique():
    sub = sc[sc['objective'] == obj].sort_values('n_new')
    improvement = -sub['mean_distance'].diff() / 1000
    ax.plot(sub['n_new'].values[1:], improvement.values[1:], marker='o', markersize=5,
            label=obj_labels[obj], color=obj_colors[obj], linewidth=2)

ax.set_xlabel('Number of New Centers')
ax.set_ylabel('Marginal Improvement (km)')
ax.set_title('Diminishing Returns: Mean Distance Reduction per Additional Center')
ax.legend()
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.axhline(0, color='gray', ls='-', lw=0.5)
plt.tight_layout()
plt.show()

### 4.3 Objective Comparison at Budget = 5

In [ ]:
compare = sc[sc['n_new'] == 5].copy()
compare['Objective'] = compare['objective'].map(obj_labels)
compare['Mean (km)'] = (compare['mean_distance'] / 1000).round(1)
compare['Max (km)'] = (compare['max_distance'] / 1000).round(1)
compare['P90 (km)'] = (compare['p90_distance'] / 1000).round(1)
compare['Coverage (%)'] = compare['pct_covered_15min'].round(1)
compare[['Objective', 'Mean (km)', 'Max (km)', 'P90 (km)', 'Coverage (%)']].reset_index(drop=True)

In [ ]:
metrics = ['Mean (km)', 'Max (km)', 'P90 (km)']
x = np.arange(len(compare))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(metrics):
    bars = ax.bar(x + i * width, compare[m].values, width, label=m, alpha=0.85)

ax.set_xlabel('Objective')
ax.set_ylabel('Distance (km)')
ax.set_title('Distance Metrics Comparison at Budget = 5 New Centers')
ax.set_xticks(x + width)
ax.set_xticklabels(compare['Objective'].values)
ax.legend()
plt.tight_layout()
plt.show()

### 4.4 Coverage vs Mean Distance Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for obj in sc['objective'].unique():
    sub = sc[(sc['objective'] == obj) & (sc['n_new'] > 0)].sort_values('n_new')
    ax.scatter(sub['mean_distance'] / 1000, sub['pct_covered_15min'],
               c=obj_colors[obj], label=obj_labels[obj], s=sub['n_new'] * 15 + 20,
               edgecolors='white', linewidth=0.5, alpha=0.8, zorder=3)

ax.set_xlabel('Mean Distance (km)')
ax.set_ylabel('15-Min Walk Coverage (%)')
ax.set_title('Trade-off: Coverage vs Mean Distance\n(bubble size = number of new centers)')
ax.legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Key Takeaways

1. **Current gap is severe** - With only 5 existing centers, most block groups are far beyond a 15-minute walk.
2. **Diminishing returns around 4-6 new centers** - Adding centers beyond this point yields progressively smaller improvements.
3. **Vulnerability-weighted objective is the sweet spot** - It achieves nearly identical mean distances to pure efficiency objectives while directing coverage toward heat-vulnerable populations.
4. **Poverty and lack of vehicle access are correlated** - These indicators compound vulnerability in the same block groups.
5. **Coverage vs distance trade-off** - The coverage-maximizing objective clusters centers in dense areas (high coverage) but leaves outlying communities underserved (higher mean distance).